# Per-File LLM Generation from IR2

Pipeline ini membaca IR2 JSON dan men-generate **satu file per LLM call** (attempt terpisah).

## Arsitektur

```
IR2.json
  └─ tasks[]          ← N task descriptor
       │
       ▼ (per task, loop)
  build_system_prompt(task, sharedContext)   ← fokus hanya file ini
  build_user_message(task)                   ← spec 1 file saja
       │
       ▼
  call_ollama(model, system, user)           ← 1 LLM call / file
       │
       ▼
  validate_content(path, content)            ← cek export, import, dll
       │
       ├── OK  → write_file(path, content)
       └── ERR → retry (max N kali) atau simpan stub kosong
```

## Keuntungan vs single-prompt
| Aspek | Single prompt | Per-file prompt |
|-------|---------------|-----------------|
| Token output | ~8.000+ (semua file) | ~150–1.050 / call |
| Risiko truncation | Tinggi pada model 7b | Tidak ada |
| Retry granularity | Harus ulang semua | Ulang 1 file saja |
| Konteks prompt | Sangat panjang | Ramping & fokus |

## 0. Konfigurasi

In [ ]:
import json
import os
import re
import time
import urllib.request
import urllib.error
from pathlib import Path
from typing import Optional

# ── Paths ──────────────────────────────────────────────────────────────────
IR2_PATH      = Path("../Car-Wash_IR/ir2.json")         # path ke IR2 hasil parser
OUTPUT_DIR    = Path("../generated_app/CarWashApp/src")  # root src output

# ── LLM Config (Ollama) ────────────────────────────────────────────────────
OLLAMA_URL    = "http://localhost:11434/api/chat"   # endpoint Ollama lokal
MODEL         = "qwen2.5-coder:7b"
MAX_RETRIES   = 2          # jumlah retry jika output tidak valid
REQUEST_TIMEOUT = 120      # detik

print(f"IR2   : {IR2_PATH.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")
print(f"Model : {MODEL}")

## 1. Load IR2

In [ ]:
with open(IR2_PATH, encoding="utf-8") as f:
    ir2 = json.load(f)

project        = ir2["project"]
stack          = ir2["stack"]
shared_context = ir2["sharedContext"]
tasks          = ir2["tasks"]

roles          = shared_context["roles"]
default_routes = shared_context["defaultRoutesPerRole"]
state_schema   = shared_context["stateSchema"]
all_routes     = shared_context["allRoutes"]

print(f"Project  : {project}")
print(f"Roles    : {[r['value'] for r in roles]}")
print(f"Total files to generate: {len(tasks)}")
for i, t in enumerate(tasks, 1):
    print(f"  {i:2d}. {t['path']}")

## 2. Prompt Builder — per file

Setiap task dalam IR2 punya field:
- `description` — tujuan file
- `path` — path output
- `group` — kategori (shared / module:X)
- `input` — data kontekstual (roles, stateSchema, routes, dll)
- `schema.exports` — nama export yang harus ada
- `constraints` — batas baris, imports, role_guard, state_key

System prompt berisi konteks proyek + aturan file type yang relevan.  
User message hanya berisi spec satu file.

In [ ]:
# ── Helper: deteksi file type dari path / group ────────────────────────────
def detect_file_type(task: dict) -> str:
    path  = task["path"]
    group = task.get("group", "")
    if "globalState" in path:          return "global_state"
    if "Layout"      in path:          return "layout"
    if "ProtectedRoute" in path:       return "protected_route"
    if "LoginPage"   in path:          return "login_page"
    if path == "src/App.tsx":          return "app_root"
    if group.startswith("module:"):    return "dynamic_page"
    return "unknown"


# ── System prompt: konteks proyek + aturan file type ──────────────────────
def build_system_prompt(task: dict) -> str:
    ftype = detect_file_type(task)
    stack_line = (
        f"Stack: {stack['framework']} + {stack['language']} + "
        f"{stack['router']} + {stack['styling']} + {stack['stateLib']}. "
        "Tailwind CSS only — no inline style objects."
    )

    # Shared context ringkas — hanya info yang dibutuhkan semua file
    roles_str   = json.dumps(roles, ensure_ascii=False)
    routes_str  = json.dumps(default_routes, ensure_ascii=False)
    schema_str  = json.dumps(list(state_schema.keys()), ensure_ascii=False)

    shared_block = (
        f"PROJECT: {project}\n"
        f"ROLES: {roles_str}\n"
        f"DEFAULT_ROUTES: {routes_str}\n"
        f"WORKFLOW_STATE_KEYS: {schema_str}\n"
        f"STATE_IMPORT: src/shared/state/globalState.ts — DO NOT regenerate this file.\n"
        f"  useGlobalState() → {{ isAuthenticated, role, userName, token, workflowState, "
        f"login(userName,role), logout(), updateWorkflow(key,value) }}"
    )

    # Aturan spesifik per file type
    rules = {
        "global_state": (
            "You are generating a Zustand store file (TypeScript).\n"
            "Export: `export interface GlobalState` and `export const useGlobalState`.\n"
            "Include: isAuthenticated, role, userName, token, workflowState.\n"
            "Include actions: login(), logout(), updateWorkflow().\n"
            "Use zustand/middleware persist. Max 120 lines."
        ),
        "app_root": (
            "You are generating src/App.tsx (BrowserRouter root).\n"
            "Register ALL routes as flat <Route> children inside <Routes>.\n"
            "CRITICAL: NEVER put <Route> inside conditional JSX {condition && <Route...>}.\n"
            "Wrap protected routes with <ProtectedRoute allowedRoles={[...]}> via element prop.\n"
            "Add one /:role/complete route per role. Add catch-all '*' with 404."
        ),
        "login_page": (
            "You are generating the login page (TSX).\n"
            "Centered card: 'userName' text input + 'role' <select> from roles list.\n"
            "On submit: useGlobalState().login(userName, role) → navigate(defaultRoutesPerRole[role]).\n"
            "Disable submit when userName is empty or role not selected.\n"
            "No external UI library — use Tailwind CSS only."
        ),
        "layout": (
            "You are generating the Layout shell (TSX).\n"
            "Sticky navbar: project title + role badge (useGlobalState().role) + Logout button.\n"
            "Logout calls logout() then navigate('/login').\n"
            "Body: <main><Outlet /></main>. No sidebar. No external UI library. Max 70 lines."
        ),
        "protected_route": (
            "You are generating the ProtectedRoute component (TSX).\n"
            "Props: { allowedRoles?: string[] }\n"
            "Logic: !isAuthenticated → <Navigate to='/login' />.\n"
            "role not in allowedRoles → <Navigate to='/login' />.\n"
            "Else → <Outlet />. Max 50 lines."
        ),
        "dynamic_page": (
            "You are generating a single BPMN task page (TSX).\n"
            "Page heading = taskName. White card layout with Tailwind.\n"
            "uiType='form'   → labeled inputs + Submit button.\n"
            "uiType='action' → status card + Proceed/Complete button.\n"
            "asyncFlag=true  → show isLoading + error useState.\n"
            "conditionalRoutes != null → render radio/select for branch choice before navigating.\n"
            "waitingFor != null → show 'Waiting for <event>...' + Continue button.\n"
            "On completion: updateWorkflow('stateKey', true) → navigate(nextRoute).\n"
            "DO NOT wrap in Layout — it is already in the router.\n"
            "DO NOT import lucide-react or any icon library. Max 110 lines."
        ),
        "unknown": "You are generating a React/TypeScript file.",
    }

    output_contract = (
        "OUTPUT CONTRACT:\n"
        "Return ONLY the raw TypeScript/TSX source code.\n"
        "No markdown fences. No prose. No JSON wrapper. Just the file content."
    )

    return "\n\n".join([
        rules.get(ftype, rules["unknown"]),
        shared_block,
        output_contract,
    ])


# ── User message: spec satu file saja ─────────────────────────────────────
def build_user_message(task: dict) -> str:
    lines = []
    lines.append(f"Generate file: {task['path']}")
    lines.append(f"Goal: {task['description']}")

    inp = task.get("input", {})
    if inp:
        slim = {k: v for k, v in inp.items() if v is not None and v != [] and v != {}}
        if slim:
            lines.append(f"Input data: {json.dumps(slim, ensure_ascii=False)}")

    sch = task.get("schema", {})
    if sch.get("exports"):
        lines.append(f"Must export: {sch['exports']}")
    if sch.get("CRITICAL"):
        lines.append(f"CRITICAL: {sch['CRITICAL']}")

    cst = task.get("constraints", {})
    for field in ("role_guard", "state_key", "async_pattern", "max_lines"):
        val = cst.get(field)
        if val:
            lines.append(f"{field}: {val}")

    lines.append("\nReturn only the source code.")
    return "\n".join(lines)


# ── Preview prompt untuk 1 task ────────────────────────────────────────────
sample = tasks[0]   # globalState.ts
print("=== SYSTEM PROMPT ===")
print(build_system_prompt(sample)[:800], "...")
print("\n=== USER MESSAGE ===")
print(build_user_message(sample))

## 3. LLM Call — Ollama

In [ ]:
def call_ollama(system_prompt: str, user_message: str,
                model: str = MODEL, timeout: int = REQUEST_TIMEOUT) -> str:
    """
    Kirim satu prompt ke Ollama dan kembalikan teks output.
    Gunakan Ollama /api/chat dengan format messages.
    """
    payload = {
        "model": model,
        "stream": False,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
    }

    req = urllib.request.Request(
        OLLAMA_URL,
        data    = json.dumps(payload).encode(),
        headers = {"Content-Type": "application/json"},
        method  = "POST",
    )

    with urllib.request.urlopen(req, timeout=timeout) as resp:
        result = json.loads(resp.read())

    return result["message"]["content"].strip()


def strip_fences(text: str) -> str:
    """Hapus markdown code fences jika LLM menambahkannya."""
    text = re.sub(r'^```[\w]*\s*\n?', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n?```\s*$',       '', text, flags=re.MULTILINE)
    return text.strip()


print("Fungsi call_ollama siap.")

## 4. Validasi Output per File

In [ ]:
def validate_content(task: dict, content: str) -> list[str]:
    """
    Validasi konten yang di-generate LLM untuk satu file.
    Kembalikan list of issue strings (kosong = valid).
    """
    issues = []
    path   = task["path"]

    if not content or len(content.strip()) < 20:
        issues.append("EMPTY_CONTENT: output terlalu pendek")
        return issues  # tidak perlu cek lanjut

    # Cek export yang diwajibkan
    exports = task.get("schema", {}).get("exports", [])
    for exp in exports:
        # Ambil nama identifier saja untuk pencarian
        ident = re.search(r'\b(\w+)\s*[:(]', exp)
        if ident and ident.group(1) not in content:
            issues.append(f"MISSING_EXPORT: '{ident.group(1)}' tidak ada di output")

    # Cek globalState tidak di-regenerate
    if "globalState" not in path and "globalState" in content:
        if ("export const useGlobalState" in content
                or "export interface GlobalState" in content):
            issues.append("FORBIDDEN: men-regenerate globalState.ts")

    # Cek App.tsx tidak pakai conditional <Route>
    if path == "src/App.tsx":
        if re.search(r'\{.*&&.*<Route', content):
            issues.append("FORBIDDEN: <Route> di dalam conditional JSX")

    # Cek dynamic page tidak re-import Layout
    ftype = detect_file_type(task)
    if ftype == "dynamic_page" and "import" in content and "Layout" in content:
        issues.append("WARN: dynamic page mengimport Layout (harusnya tidak)")

    return issues


print("Fungsi validate_content siap.")

## 5. Write File ke Disk

In [ ]:
def write_output_file(relative_path: str, content: str, base_dir: Path = OUTPUT_DIR):
    """
    Tulis konten ke disk. Path adalah relatif terhadap root proyek,
    dimulai dengan 'src/'. base_dir adalah parent dari 'src/'.
    """
    # relative_path contoh: "src/modules/Customer/pages/ChoosesWashPage.tsx"
    # base_dir adalah OUTPUT_DIR = generated_app/CarWashApp/src  (sudah termasuk src/)
    # Jadi perlu strip 'src/' prefix dari relative_path
    rel = relative_path
    if rel.startswith("src/"):
        rel = rel[len("src/"):]

    target = base_dir / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return target


print("Fungsi write_output_file siap.")

## 6. Main Loop — Generate Setiap File dalam Attempt Terpisah

Setiap task di-generate secara berurutan.  
Jika validasi gagal, attempt diulang hingga `MAX_RETRIES` kali.  
Hasilnya diakumulasi di `results`.

In [ ]:
results = []  # list of dict per file

for idx, task in enumerate(tasks, 1):
    path   = task["path"]
    ftype  = detect_file_type(task)
    print(f"[{idx:2d}/{len(tasks)}] {path} ({ftype})")

    system_prompt = build_system_prompt(task)
    user_message  = build_user_message(task)

    attempt     = 0
    final_content = ""
    final_issues  = []
    status        = "pending"

    while attempt <= MAX_RETRIES:
        attempt += 1
        try:
            raw = call_ollama(system_prompt, user_message)
            content = strip_fences(raw)
            issues  = validate_content(task, content)

            final_content = content
            final_issues  = issues

            if not issues:
                status = "ok"
                break  # valid — lanjut ke file berikutnya
            else:
                print(f"   attempt {attempt}: {issues}")
                if attempt <= MAX_RETRIES:
                    # Tambahkan feedback ke user_message untuk retry
                    user_message = (
                        build_user_message(task) +
                        f"\n\nPrevious attempt had issues: {issues}. Fix them."
                    )

        except Exception as e:
            print(f"   attempt {attempt} ERROR: {e}")
            final_issues = [f"LLM_ERROR: {e}"]
            status = "error"
            time.sleep(2)

    # Tentukan status akhir
    if status != "ok":
        status = "warn" if final_content else "error"

    # Tulis file (konten valid atau partial)
    written_path = None
    if final_content:
        written_path = write_output_file(path, final_content)

    results.append({
        "idx":     idx,
        "path":    path,
        "ftype":   ftype,
        "status":  status,
        "attempts":attempt,
        "issues":  final_issues,
        "written": str(written_path) if written_path else None,
    })

    icon = "✅" if status == "ok" else "⚠️" if status == "warn" else "❌"
    print(f"   {icon} {status} (attempt {attempt})")

print("\n✅ Loop selesai.")

## 7. Ringkasan Hasil

In [ ]:
ok_count    = sum(1 for r in results if r["status"] == "ok")
warn_count  = sum(1 for r in results if r["status"] == "warn")
error_count = sum(1 for r in results if r["status"] == "error")
total       = len(results)

print(f"{'='*55}")
print(f"  HASIL GENERASI — {project}")
print(f"{'='*55}")
print(f"  Total  : {total} file")
print(f"  ✅ OK   : {ok_count}")
print(f"  ⚠️  Warn : {warn_count}")
print(f"  ❌ Error: {error_count}")
print(f"{'='*55}")

if warn_count or error_count:
    print("\nFile bermasalah:")
    for r in results:
        if r["status"] != "ok":
            print(f"  [{r['status'].upper()}] {r['path']}")
            for iss in r["issues"]:
                print(f"         → {iss}")

# Export log JSON
log_path = OUTPUT_DIR.parent / "generation_log.json"
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nLog ditulis ke: {log_path}")

## 8. (Opsional) Preview Satu File Hasil Generate

In [ ]:
# Ganti index untuk melihat file lain
PREVIEW_IDX = 0   # index dalam results[]

r = results[PREVIEW_IDX]
print(f"Path   : {r['path']}")
print(f"Status : {r['status']}")
print(f"Issues : {r['issues']}")
print(f"Written: {r['written']}")

if r["written"]:
    content = Path(r["written"]).read_text(encoding="utf-8")
    print(f"\n{'─'*60}")
    print(content[:2000])
    if len(content) > 2000:
        print(f"... ({len(content)} chars total)")

## 9. (Opsional) Regenerate File Tertentu

Jalankan cell ini untuk mengulang generasi satu file saja tanpa menjalankan ulang seluruh loop.

In [ ]:
# Ganti path sesuai file yang ingin di-regenerate
REGEN_PATH = "src/shared/components/Layout.tsx"

task = next((t for t in tasks if t["path"] == REGEN_PATH), None)
if not task:
    print(f"Path tidak ditemukan di IR2: {REGEN_PATH}")
else:
    sys_p = build_system_prompt(task)
    usr_p = build_user_message(task)

    print(f"Regenerating: {REGEN_PATH}")
    raw     = call_ollama(sys_p, usr_p)
    content = strip_fences(raw)
    issues  = validate_content(task, content)

    print(f"Issues: {issues if issues else 'none'}")
    written = write_output_file(REGEN_PATH, content)
    print(f"Written: {written}")
    print(content[:1000])